# Phase 2: Dataset Manifest and Quality Audit

**Objective:** Build dataset manifests from the actual extracted BUSI and BUS-UCLM files. Do not create splits and do not train models.

**Tasks:**
1. Verify Phase 1 evidence
2. Discover actual dataset locations and structure
3. Implement reusable discovery and manifest logic
4. Define the primary study as binary classification: benign vs malignant
5. Retain normal images in the manifest but set `included_in_primary_task = false`
6. Never map normal images to benign
7. Handle BUSI multi-mask cases with binary union
8. Create stable sample IDs with comprehensive metadata
9. Validate images, masks, labels, and paths
10. Flag rather than silently delete problematic samples
11. Produce deterministic visual audit grids
12. Save manifests and reports

**Status Label:** `planned` -> `implemented` -> `runnable` -> `executed` -> `validated`

**Important:** This notebook must run from top to bottom after a clean-kernel restart.

In [ ]:
# Cell 1: Environment Setup and Project Root Resolution
# This cell must run first and set PROJECT_ROOT

import os
import sys
from pathlib import Path
import json
import logging
import subprocess
from datetime import datetime, timezone

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger('phase_02_manifest')

# Resolve project root
def find_project_root(start_path: Path = Path.cwd()) -> Path:
    """Find project root by looking for CausalMask-XAI.md."""
    current = start_path
    while current != current.parent:
        if (current / 'CausalMask-XAI.md').exists():
            return current
        current = current.parent
    if Path('/content/CausalMask-XAI').exists():
        return Path('/content/CausalMask-XAI')
    raise FileNotFoundError('Could not find project root (CausalMask-XAI.md)')

PROJECT_ROOT = find_project_root()
os.environ['CAUSALMASK_PROJECT_ROOT'] = str(PROJECT_ROOT)
src_path = PROJECT_ROOT / 'src'
sys.path.insert(0, str(src_path))

# Verify package structure; restore missing tracked files via git
_data_init = src_path / 'causalmask' / 'data' / '__init__.py'
if not _data_init.exists():
    print(f'WARNING: {_data_init} not found.')
    print('Colab ephemeral storage may have lost source files.')
    print('Attempting git checkout to restore tracked source files...')
    result = subprocess.run(
        ['git', 'checkout', 'HEAD', '--', 'src/'],
        cwd=str(PROJECT_ROOT), capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(f'git checkout failed: {result.stderr}')
        print('Falling back to pip install -e .')
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-e', '.'],
            cwd=str(PROJECT_ROOT), capture_output=True,
        )
    if not (src_path / 'causalmask' / 'data' / '__init__.py').exists():
        raise RuntimeError(
            f'Package restore failed. {_data_init} still missing.\n'
            f'Manually run: cd {PROJECT_ROOT} && git checkout HEAD -- src/'
        )
    print('Source files restored via git checkout.')

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'src path     = {src_path}')
print(f'Python: {sys.version}')
print(f'Working directory: {Path.cwd()}')
print(f'CAUSALMASK_PROJECT_ROOT env: {os.environ.get("CAUSALMASK_PROJECT_ROOT", "NOT SET")}')

In [ ]:
# Cell 2: Install dependencies if needed

# Check if we're in Colab
IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
    print('Detected Google Colab environment.')
except ImportError:
    print('Running in local environment.')

if IN_COLAB:
    # Install required packages
    import subprocess
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow', 'matplotlib', 'seaborn', 'tqdm', 'pandas'],
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        print(f'Warning: pip install returned {result.returncode}')
    else:
        print('Dependencies installed.')

# Verify imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm

print('All imports successful.')

In [ ]:
# Cell 3: Configuration and Seed Setting

import torch
import random

# Set deterministic seeds
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# Configuration
CONFIG = {
    'seed': SEED,
    'phase': '02',
    'phase_name': 'Dataset Manifest and Quality Audit',
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'project_root': str(PROJECT_ROOT),
    'in_colab': IN_COLAB,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'datasets': {
        'busi': {
            'extract_rel': 'data/raw/extracted/busi',
            'manifest_rel': 'data/manifests/busi_manifest_v1.parquet',
            'summary_rel': 'data/manifests/busi_manifest_summary_v1.json',
        },
        'bus_uclm': {
            'extract_rel': 'data/raw/extracted/bus_uclm',
            'manifest_rel': 'data/manifests/bus_uclm_manifest_v1.parquet',
            'summary_rel': 'data/manifests/bus_uclm_manifest_summary_v1.json',
        },
    },
    'primary_task_labels': ['benign', 'malignant'],
    'output_dirs': [
        'data/manifests',
        'reports',
        'reports/results/data_audit_grids',
        'artifacts/phases',
    ],
}

# Create output directories
for d in CONFIG['output_dirs']:
    (PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)

# Display configuration
print('Configuration:')
print(json.dumps(CONFIG, indent=2, default=str))

# Save configuration
config_path = PROJECT_ROOT / 'artifacts' / 'phases' / 'phase_02_config.json'
with open(config_path, 'w') as f:
    json.dump(CONFIG, f, indent=2, default=str)
print(f'\nConfiguration saved to: {config_path}')

In [ ]:
# Cell 4: Import CausalMask modules

from causalmask.data.manifest import (
    discover_busi_files,
    discover_bus_uclm_files,
    build_manifest,
    validate_manifest,
    save_manifest_parquet,
    save_manifest_summary,
    normalize_label,
    generate_sample_id,
    SampleRecord,
    atomic_write_parquet,
    atomic_write_json,
    compute_sha256,
)
from causalmask.reproducibility import configure_reproducibility

# Configure reproducibility
repro_info = configure_reproducibility(SEED)
print('Reproducibility configured:')
print(json.dumps(repro_info, indent=2))

## Google Drive Persistence (Optional)

This section provides optional Google Drive backup for durable research artifacts.
The main Phase 2 workflow operates entirely in `/content/CausalMask-XAI` regardless of Drive status.

In [ ]:
# Cell 5: Google Drive Persistence Setup (Optional)

DRIVE_MOUNTED = False
DRIVE_BACKUP_DIR = None
BACKUP_ENABLED = False

if IN_COLAB:
    from google.colab import drive
    
    # Check if Drive is already mounted
    drive_mount_point = Path('/content/drive')
    if drive_mount_point.exists() and any(drive_mount_point.iterdir()):
        print('Drive already mounted at /content/drive')
        DRIVE_MOUNTED = True
    else:
        print('Drive not mounted. To enable backup, execute the mount cell below.')
        print('If you do not mount Drive, outputs remain temporary.')
else:
    print('Not running in Google Colab. Drive persistence not available.')

if DRIVE_MOUNTED:
    DRIVE_BACKUP_DIR = Path('/content/drive/MyDrive/CausalMask-XAI-backup')
    DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
    BACKUP_ENABLED = True
    print(f'Drive mounted and backup enabled: {DRIVE_BACKUP_DIR}')
else:
    print('Drive not mounted and outputs remain temporary.')

In [ ]:
# Cell 5b: Mount Google Drive (Optional - execute only if you want backup)

# Uncomment and execute this cell to mount Google Drive
# DRIVE_MOUNTED = False
# try:
#     from google.colab import drive
#     drive.mount('/content/drive')
#     DRIVE_MOUNTED = True
#     DRIVE_BACKUP_DIR = Path('/content/drive/MyDrive/CausalMask-XAI-backup')
#     DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
#     BACKUP_ENABLED = True
#     print(f'Drive mounted and backup enabled: {DRIVE_BACKUP_DIR}')
# except Exception as e:
#     print(f'Drive mount failed: {e}')
#     print('Outputs remain temporary.')

In [ ]:
# Cell 6: Safe Backup Functions

import shutil
from datetime import datetime

def safe_backup_to_drive(
    project_root: Path,
    backup_dir: Path,
    include_archives: bool = False,
    dry_run: bool = False,
) -> dict:
    """Copy durable research artifacts from project to Drive backup directory.
    
    Args:
        project_root: Source project directory
        backup_dir: Destination backup directory on Drive
        include_archives: If True, also backup ZIP archives (~837 MB)
        dry_run: If True, only report what would be copied
    
    Returns:
        Dictionary with backup statistics and manifest
    """
    backup_stats = {
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'source': str(project_root),
        'destination': str(backup_dir),
        'include_archives': include_archives,
        'dry_run': dry_run,
        'categories': {},
        'total_files_copied': 0,
        'total_bytes_copied': 0,
        'errors': [],
    }
    
    # Define backup categories
    backup_categories = {
        'project_spec': ['CausalMask-XAI.md', '.opencode/'],
        'notebooks': ['notebooks/'],
        'source_code': ['src/'],
        'configs': ['configs/'],
        'tests': ['tests/'],
        'scripts': ['scripts/'],
        'project_files': ['pyproject.toml', '.gitignore'],
        'data_readme': ['data/README.md', 'data/raw/dataset_sources.json'],
        'manifests': ['data/manifests/'],
        'splits': ['data/splits/'],
        'reports': ['reports/'],
        'phase_status': ['artifacts/phases/'],
    }
    
    # Optional archive backup
    if include_archives:
        backup_categories['archives'] = ['data/raw/archives/']
    
    print(f'Backup target: {backup_dir}')
    print(f'Dry run: {dry_run}')
    print()
    
    for category, paths in backup_categories.items():
        category_files = 0
        category_bytes = 0
        
        for rel_path in paths:
            src_path = project_root / rel_path
            
            if not src_path.exists():
                print(f'  [{category}] SKIP (not found): {rel_path}')
                continue
            
            if src_path.is_file():
                # Single file
                dst_path = backup_dir / rel_path
                dst_path.parent.mkdir(parents=True, exist_ok=True)
                
                # Check if newer
                if dst_path.exists() and dst_path.stat().st_mtime >= src_path.stat().st_mtime:
                    print(f'  [{category}] SKIP (up-to-date): {rel_path}')
                    continue
                
                if not dry_run:
                    shutil.copy2(src_path, dst_path)
                
                category_files += 1
                category_bytes += src_path.stat().st_size
                print(f'  [{category}] COPY: {rel_path} ({src_path.stat().st_size:,} bytes)')
            
            elif src_path.is_dir():
                # Directory - copy recursively
                for file_path in src_path.rglob('*'):
                    if file_path.is_file():
                        rel_file = file_path.relative_to(project_root)
                        dst_path = backup_dir / rel_file
                        dst_path.parent.mkdir(parents=True, exist_ok=True)
                        
                        # Skip if up-to-date
                        if dst_path.exists() and dst_path.stat().st_mtime >= file_path.stat().st_mtime:
                            continue
                        
                        # Skip checkpoint directories
                        if '.ipynb_checkpoints' in str(file_path):
                            continue
                        
                        if not dry_run:
                            shutil.copy2(file_path, dst_path)
                        
                        category_files += 1
                        category_bytes += file_path.stat().st_size
                
                print(f'  [{category}] COPY ({category_files} files): {rel_path}/')
        
        backup_stats['categories'][category] = {
            'files_copied': category_files,
            'bytes_copied': category_bytes,
        }
        backup_stats['total_files_copied'] += category_files
        backup_stats['total_bytes_copied'] += category_bytes
    
    print(f'\nBackup complete: {backup_stats["total_files_copied"]} files, '
          f'{backup_stats["total_bytes_copied"]:,} bytes')
    
    # Save backup manifest
    if not dry_run:
        manifest_path = backup_dir / 'backup_manifest.json'
        manifest_path.parent.mkdir(parents=True, exist_ok=True)
        with open(manifest_path, 'w') as f:
            json.dump(backup_stats, f, indent=2, default=str)
        print(f'Backup manifest saved: {manifest_path}')
    
    return backup_stats

print('Backup functions defined.')

## Phase 1 Verification

Verify that Phase 1 evidence exists and is consistent.

In [ ]:
# Cell 7: Verify Phase 1 Evidence

print('Verifying Phase 1 evidence...\n')

phase1_checks = {
    'dataset_sources_json': PROJECT_ROOT / 'data' / 'raw' / 'dataset_sources.json',
    'archives_dir': PROJECT_ROOT / 'data' / 'raw' / 'archives',
    'extracted_dir': PROJECT_ROOT / 'data' / 'raw' / 'extracted',
    'phase_01_status': PROJECT_ROOT / 'artifacts' / 'phases' / 'phase_01_status.json',
    'data_download_report': PROJECT_ROOT / 'reports' / 'data_download_report.md',
}

phase1_evidence = {}
for check_name, path in phase1_checks.items():
    exists = path.exists()
    phase1_evidence[check_name] = {
        'path': str(path),
        'exists': exists,
    }
    status = 'PASS' if exists else 'FAIL'
    print(f'  [{status}] {check_name}: {path}')

# Check archives
if phase1_evidence['archives_dir']['exists']:
    archives_dir = PROJECT_ROOT / 'data' / 'raw' / 'archives'
    archive_files = list(archives_dir.glob('*'))
    archive_files = [f for f in archive_files if f.name != '.gitkeep']
    phase1_evidence['archives_dir']['file_count'] = len(archive_files)
    phase1_evidence['archives_dir']['files'] = [f.name for f in archive_files]
    print(f'\nArchives found: {len(archive_files)}')
    for f in archive_files:
        print(f'  - {f.name} ({f.stat().st_size:,} bytes)')

# Check extracted
if phase1_evidence['extracted_dir']['exists']:
    extracted_dir = PROJECT_ROOT / 'data' / 'raw' / 'extracted'
    extracted_items = list(extracted_dir.iterdir())
    extracted_items = [f for f in extracted_items if f.name != '.gitkeep']
    phase1_evidence['extracted_dir']['item_count'] = len(extracted_items)
    phase1_evidence['extracted_dir']['items'] = [f.name for f in extracted_items]
    print(f'\nExtracted items: {len(extracted_items)}')
    for item in extracted_items:
        item_type = 'DIR' if item.is_dir() else 'FILE'
        print(f'  [{item_type}] {item.name}')

# Check phase 01 status
if phase1_evidence['phase_01_status']['exists']:
    with open(PROJECT_ROOT / 'artifacts' / 'phases' / 'phase_01_status.json') as f:
        phase1_status = json.load(f)
    phase1_evidence['phase_01_status']['status'] = phase1_status.get('status_label', 'unknown')
    print(f'\nPhase 01 status: {phase1_status.get("status_label", "unknown")}')

# Check dataset_sources.json
if phase1_evidence['dataset_sources_json']['exists']:
    with open(PROJECT_ROOT / 'data' / 'raw' / 'dataset_sources.json') as f:
        sources = json.load(f)
    phase1_evidence['dataset_sources_json']['sources'] = sources
    print(f'\nDataset sources:')
    print(json.dumps(sources, indent=2))

# Summary
all_phase1_ok = all(v['exists'] for v in phase1_evidence.values())
print(f'\nPhase 1 verification: {"PASS" if all_phase1_ok else "FAIL"}')
if not all_phase1_ok:
    print('WARNING: Some Phase 1 evidence is missing. Proceeding with caution.')

## Dataset Discovery

Discover actual dataset locations and structure from extracted files.

In [ ]:
# Cell 8: Discover Actual Dataset Locations

print('Discovering actual dataset locations...\n')

extracted_base = PROJECT_ROOT / 'data' / 'raw' / 'extracted'
discovered_datasets = {}

if extracted_base.exists():
    # List all items in extracted directory
    all_items = [f for f in extracted_base.iterdir() if f.name != '.gitkeep']
    print(f'Items in extracted directory: {len(all_items)}')
    for item in all_items:
        item_type = 'DIR' if item.is_dir() else 'FILE'
        print(f'  [{item_type}] {item.name}')
    
    # Try to identify BUSI and BUS-UCLM
    for item in all_items:
        if not item.is_dir():
            continue
        
        # Check if this directory contains label subdirectories
        label_dirs = []
        for sub in item.iterdir():
            if sub.is_dir() and sub.name.lower() in ('benign', 'malignant', 'normal'):
                label_dirs.append(sub)
        
        if label_dirs:
            print(f'\n  Found label directories in: {item.name}')
            for ld in label_dirs:
                files = list(ld.glob('*.png'))
                print(f'    {ld.name}: {len(files)} files')
            
            # Try to identify dataset by content
            # BUSI typically has: Dataset_BUSI_with_GT/benign, malignant, normal
            # BUS-UCLM typically has: bus_uclm_separated/benign, malignant, normal
            # or: BUS-UCLM Breast ultrasound lesion segmentation dataset/benign, malignant, normal
            
            if item.name == 'Dataset_BUSI_with_GT':
                discovered_datasets['busi'] = {
                    'path': item,
                    'label_dirs': {ld.name: ld for ld in label_dirs},
                    'source': 'directory_name_match',
                }
            elif 'bus_uclm' in item.name.lower() or 'breast ultrasound' in item.name.lower():
                discovered_datasets['bus_uclm'] = {
                    'path': item,
                    'label_dirs': {ld.name: ld for ld in label_dirs},
                    'source': 'directory_name_match',
                }
            else:
                # Unknown dataset - try to identify by file count or other heuristics
                print(f'    Unknown dataset: {item.name}')
                discovered_datasets[f'unknown_{item.name}'] = {
                    'path': item,
                    'label_dirs': {ld.name: ld for ld in label_dirs},
                    'source': 'unknown',
                }
    
    # If no datasets found, try recursive search
    if not discovered_datasets:
        print('\nNo datasets found at top level. Searching recursively...')
        for item in all_items:
            if item.is_dir():
                for sub in item.rglob('*'):
                    if sub.is_dir() and sub.name.lower() in ('benign', 'malignant', 'normal'):
                        print(f'  Found label dir: {sub}')

# Report discovery
print(f'\nDiscovered datasets: {len(discovered_datasets)}')
for name, info in discovered_datasets.items():
    print(f'  {name}: {info["path"]} ({info["source"]})')

# Verify both datasets found
if 'busi' not in discovered_datasets:
    print('\nERROR: BUSI dataset not found!')
    print('Expected directory: Dataset_BUSI_with_GT with benign/, malignant/, normal/ subdirectories')
    PHASE2_BLOCKED = True
else:
    PHASE2_BLOCKED = False

if 'bus_uclm' not in discovered_datasets:
    print('\nWARNING: BUS-UCLM dataset not found.')
    print('Expected directory: bus_uclm_separated/ or BUS-UCLM Breast ultrasound... with benign/, malignant/, normal/ subdirectories')
    # Don't block for BUS-UCLM - it's external validation only

print(f'\nPhase 2 blocked: {PHASE2_BLOCKED}')

In [ ]:
# Cell 9: Inspect BUSI Structure in Detail

if 'busi' in discovered_datasets:
    busi_info = discovered_datasets['busi']
    print('BUSI Dataset Structure:')
    print(f'  Root: {busi_info["path"]}')
    
    for label, label_dir in busi_info['label_dirs'].items():
        files = sorted(label_dir.glob('*.png'))
        
        # Separate images and masks
        images = [f for f in files if '_mask' not in f.name.lower()]
        masks = [f for f in files if '_mask' in f.name.lower()]
        
        print(f'\n  {label}/:')
        print(f'    Total files: {len(files)}')
        print(f'    Images: {len(images)}')
        print(f'    Masks: {len(masks)}')
        
        # Show filename patterns
        if images:
            print(f'\n    Sample image filenames:')
            for f in images[:5]:
                print(f'      {f.name}')
        
        if masks:
            print(f'\n    Sample mask filenames:')
            for f in masks[:10]:
                print(f'      {f.name}')
            
            # Check for multi-mask patterns
            mask_stems = [f.stem for f in masks]
            multi_mask_stems = []
            for stem in set(mask_stems):
                count = mask_stems.count(stem)
                if count > 1:
                    multi_mask_stems.append((stem, count))
            
            if multi_mask_stems:
                print(f'\n    Multi-mask samples found:')
                for stem, count in multi_mask_stems[:5]:
                    print(f'      {stem}: {count} masks')

else:
    print('BUSI dataset not discovered. Cannot inspect structure.')

In [ ]:
# Cell 10: Inspect BUS-UCLM Structure in Detail

if 'bus_uclm' in discovered_datasets:
    bus_uclm_info = discovered_datasets['bus_uclm']
    print('BUS-UCLM Dataset Structure:')
    print(f'  Root: {bus_uclm_info["path"]}')
    
    for label, label_dir in bus_uclm_info['label_dirs'].items():
        files = sorted(label_dir.glob('*.png'))
        
        # Separate images and masks
        images = [f for f in files if '_mask' not in f.name.lower()]
        masks = [f for f in files if '_mask' in f.name.lower()]
        
        print(f'\n  {label}/:')
        print(f'    Total files: {len(files)}')
        print(f'    Images: {len(images)}')
        print(f'    Masks: {len(masks)}')
        
        # Show filename patterns
        if images:
            print(f'\n    Sample image filenames:')
            for f in images[:5]:
                print(f'      {f.name}')
        
        if masks:
            print(f'\n    Sample mask filenames:')
            for f in masks[:10]:
                print(f'      {f.name}')
else:
    print('BUS-UCLM dataset not discovered. Cannot inspect structure.')

## Dataset Discovery Functions

Use the manifest module to discover files from actual dataset locations.

In [ ]:
# Cell 11: Discover BUSI Files

print('Discovering BUSI files...\n')

if 'busi' in discovered_datasets:
    busi_extract_dir = discovered_datasets['busi']['path']
    busi_samples = discover_busi_files(busi_extract_dir, dataset_name='busi')
    
    print(f'Found {len(busi_samples)} BUSI samples\n')
    
    # Show label distribution
    label_counts = {}
    for s in busi_samples:
        label = s['normalized_label']
        label_counts[label] = label_counts.get(label, 0) + 1
    
    print('Label distribution:')
    for label, count in sorted(label_counts.items()):
        print(f'  {label}: {count}')
    
    # Show multi-mask samples
    multi_mask_samples = [s for s in busi_samples if len(s['mask_paths']) > 1]
    if multi_mask_samples:
        print(f'\nMulti-mask samples: {len(multi_mask_samples)}')
        for s in multi_mask_samples[:5]:
            print(f'  {s["image_path"].name}: {len(s["mask_paths"])} masks')
    
    # Show sample paths
    print(f'\nSample paths (first 5):')
    for s in busi_samples[:5]:
        print(f'  Image: {s["image_path"]}')
        print(f'  Masks: {s["mask_paths"]}')
        print()
else:
    print('BUSI dataset not available. Cannot discover files.')
    busi_samples = []

In [ ]:
# Cell 12: Discover BUS-UCLM Files

print('Discovering BUS-UCLM files...\n')

if 'bus_uclm' in discovered_datasets:
    bus_uclm_extract_dir = discovered_datasets['bus_uclm']['path']
    bus_uclm_samples = discover_bus_uclm_files(bus_uclm_extract_dir, dataset_name='bus_uclm')
    
    print(f'Found {len(bus_uclm_samples)} BUS-UCLM samples\n')
    
    # Show label distribution
    label_counts = {}
    for s in bus_uclm_samples:
        label = s['normalized_label']
        label_counts[label] = label_counts.get(label, 0) + 1
    
    print('Label distribution:')
    for label, count in sorted(label_counts.items()):
        print(f'  {label}: {count}')
    
    # Show multi-mask samples
    multi_mask_samples = [s for s in bus_uclm_samples if len(s['mask_paths']) > 1]
    if multi_mask_samples:
        print(f'\nMulti-mask samples: {len(multi_mask_samples)}')
        for s in multi_mask_samples[:5]:
            print(f'  {s["image_path"].name}: {len(s["mask_paths"])} masks')
    
    # Show sample paths
    print(f'\nSample paths (first 5):')
    for s in bus_uclm_samples[:5]:
        print(f'  Image: {s["image_path"]}')
        print(f'  Masks: {s["mask_paths"]}')
        print()
else:
    print('BUS-UCLM dataset not available. Cannot discover files.')
    bus_uclm_samples = []

## Build Manifests

Create comprehensive manifests for both datasets with all required fields.
BUSI multi-mask cases are handled with binary union.

In [ ]:
# Cell 13: Build BUSI Manifest

print('Building BUSI manifest...\n')

busi_records = build_manifest(
    samples=busi_samples,
    project_root=PROJECT_ROOT,
    dataset_name='busi',
    patient_id_prefix='busi_',
)

print(f'Built {len(busi_records)} BUSI records\n')

# Show label distribution
label_counts = {}
for r in busi_records:
    label = r.normalized_label
    label_counts[label] = label_counts.get(label, 0) + 1

print('Label distribution:')
for label, count in sorted(label_counts.items()):
    print(f'  {label}: {count}')

# Show primary task count
primary_count = sum(1 for r in busi_records if r.included_in_primary_task)
print(f'\nPrimary task samples (benign + malignant): {primary_count}')
print(f'Normal samples (excluded from primary task): {len(busi_records) - primary_count}')

# Show multi-mask samples
multi_mask_records = [r for r in busi_records if 'multi_mask' in str(r.quality_flags)]
if multi_mask_records:
    print(f'\nMulti-mask samples: {len(multi_mask_records)}')

In [ ]:
# Cell 14: Build BUS-UCLM Manifest

print('Building BUS-UCLM manifest...\n')

bus_uclm_records = build_manifest(
    samples=bus_uclm_samples,
    project_root=PROJECT_ROOT,
    dataset_name='bus_uclm',
    patient_id_prefix='bus_uclm_',
)

print(f'Built {len(bus_uclm_records)} BUS-UCLM records\n')

# Show label distribution
label_counts = {}
for r in bus_uclm_records:
    label = r.normalized_label
    label_counts[label] = label_counts.get(label, 0) + 1

print('Label distribution:')
for label, count in sorted(label_counts.items()):
    print(f'  {label}: {count}')

# Show primary task count
primary_count = sum(1 for r in bus_uclm_records if r.included_in_primary_task)
print(f'\nPrimary task samples (benign + malignant): {primary_count}')
print(f'Normal samples (excluded from primary task): {len(bus_uclm_records) - primary_count}')

## Validate Manifests

Run comprehensive validation checks on both manifests.

In [ ]:
# Cell 15: Validate BUSI Manifest

print('Validating BUSI manifest...\n')

busi_validation = validate_manifest(busi_records)

print('Validation Summary:')
print(json.dumps(busi_validation['issue_counts'], indent=2))

# Show specific issues
if busi_validation['issues']['duplicate_sample_ids']:
    print(f'\nDuplicate sample IDs: {len(busi_validation["issues"]["duplicate_sample_ids"])}')

if busi_validation['issues']['abnormal_without_mask']:
    print(f'Abnormal cases without mask: {len(busi_validation["issues"]["abnormal_without_mask"])}')
    for sid in busi_validation['issues']['abnormal_without_mask'][:5]:
        print(f'  - {sid}')

if busi_validation['issues']['quality_flagged']:
    print(f'\nQuality flagged samples: {len(busi_validation["issues"]["quality_flagged"])}')
    # Show flag distribution
    flag_counts = {}
    for item in busi_validation['issues']['quality_flagged']:
        for flag in item['flags']:
            flag_counts[flag] = flag_counts.get(flag, 0) + 1
    print('Flag distribution:')
    for flag, count in sorted(flag_counts.items()):
        print(f'  {flag}: {count}')

In [ ]:
# Cell 16: Validate BUS-UCLM Manifest

print('Validating BUS-UCLM manifest...\n')

bus_uclm_validation = validate_manifest(bus_uclm_records)

print('Validation Summary:')
print(json.dumps(bus_uclm_validation['issue_counts'], indent=2))

# Show specific issues
if bus_uclm_validation['issues']['duplicate_sample_ids']:
    print(f'\nDuplicate sample IDs: {len(bus_uclm_validation["issues"]["duplicate_sample_ids"])}')

if bus_uclm_validation['issues']['abnormal_without_mask']:
    print(f'Abnormal cases without mask: {len(bus_uclm_validation["issues"]["abnormal_without_mask"])}')
    for sid in bus_uclm_validation['issues']['abnormal_without_mask'][:5]:
        print(f'  - {sid}')

if bus_uclm_validation['issues']['quality_flagged']:
    print(f'\nQuality flagged samples: {len(bus_uclm_validation["issues"]["quality_flagged"])}')
    # Show flag distribution
    flag_counts = {}
    for item in bus_uclm_validation['issues']['quality_flagged']:
        for flag in item['flags']:
            flag_counts[flag] = flag_counts.get(flag, 0) + 1
    print('Flag distribution:')
    for flag, count in sorted(flag_counts.items()):
        print(f'  {flag}: {count}')

## Save Manifests

Save manifests as Parquet files and summaries as JSON.
Uses atomic writes to prevent corruption.

In [ ]:
# Cell 17: Save BUSI Manifest (Atomic Write)

print('Saving BUSI manifest...\n')

busi_manifest_path = PROJECT_ROOT / CONFIG['datasets']['busi']['manifest_rel']
busi_summary_path = PROJECT_ROOT / CONFIG['datasets']['busi']['summary_rel']

# Check if manifest already exists
if busi_manifest_path.exists():
    print(f'WARNING: Manifest already exists: {busi_manifest_path}')
    print('Creating new version to avoid overwriting.')
    # Create new version
    version = 1
    while busi_manifest_path.exists():
        version += 1
        busi_manifest_path = PROJECT_ROOT / f'data/manifests/busi_manifest_v{version}.parquet'
        busi_summary_path = PROJECT_ROOT / f'data/manifests/busi_manifest_summary_v{version}.json'
    CONFIG['datasets']['busi']['manifest_rel'] = str(busi_manifest_path.relative_to(PROJECT_ROOT))
    CONFIG['datasets']['busi']['summary_rel'] = str(busi_summary_path.relative_to(PROJECT_ROOT))

atomic_write_parquet(busi_records, busi_manifest_path)
atomic_write_json(busi_validation, busi_records, 'busi', busi_summary_path)

print(f'Manifest saved to: {busi_manifest_path}')
print(f'Summary saved to: {busi_summary_path}')

In [ ]:
# Cell 18: Save BUS-UCLM Manifest (Atomic Write)

print('Saving BUS-UCLM manifest...\n')

bus_uclm_manifest_path = PROJECT_ROOT / CONFIG['datasets']['bus_uclm']['manifest_rel']
bus_uclm_summary_path = PROJECT_ROOT / CONFIG['datasets']['bus_uclm']['summary_rel']

# Check if manifest already exists
if bus_uclm_manifest_path.exists():
    print(f'WARNING: Manifest already exists: {bus_uclm_manifest_path}')
    print('Creating new version to avoid overwriting.')
    # Create new version
    version = 1
    while bus_uclm_manifest_path.exists():
        version += 1
        bus_uclm_manifest_path = PROJECT_ROOT / f'data/manifests/bus_uclm_manifest_v{version}.parquet'
        bus_uclm_summary_path = PROJECT_ROOT / f'data/manifests/bus_uclm_manifest_summary_v{version}.json'
    CONFIG['datasets']['bus_uclm']['manifest_rel'] = str(bus_uclm_manifest_path.relative_to(PROJECT_ROOT))
    CONFIG['datasets']['bus_uclm']['summary_rel'] = str(bus_uclm_summary_path.relative_to(PROJECT_ROOT))

atomic_write_parquet(bus_uclm_records, bus_uclm_manifest_path)
atomic_write_json(bus_uclm_validation, bus_uclm_records, 'bus_uclm', bus_uclm_summary_path)

print(f'Manifest saved to: {bus_uclm_manifest_path}')
print(f'Summary saved to: {bus_uclm_summary_path}')

In [ ]:
# Cell 19: Save Exclusions CSV

print('Saving exclusions...\n')

exclusions_path = PROJECT_ROOT / 'data' / 'manifests' / 'manifest_exclusions_v1.csv'

# Collect all exclusions
exclusions = []

# From BUSI validation
for item in busi_validation['issues']['quality_flagged']:
    exclusions.append({
        'dataset': 'busi',
        'sample_id': item['sample_id'],
        'flags': '|'.join(item['flags']),
        'reason': 'quality_flag',
    })

# From BUS-UCLM validation
for item in bus_uclm_validation['issues']['quality_flagged']:
    exclusions.append({
        'dataset': 'bus_uclm',
        'sample_id': item['sample_id'],
        'flags': '|'.join(item['flags']),
        'reason': 'quality_flag',
    })

# Save exclusions
if exclusions:
    exclusions_df = pd.DataFrame(exclusions)
    exclusions_df.to_csv(exclusions_path, index=False)
    print(f'Saved {len(exclusions)} exclusions to: {exclusions_path}')
else:
    print('No exclusions to save.')
    # Create empty file
    pd.DataFrame(columns=['dataset', 'sample_id', 'flags', 'reason']).to_csv(exclusions_path, index=False)

## Visual Audit Grids

Create deterministic visual audit grids for sample inspection.

In [ ]:
# Cell 20: Helper function for visual audit grids

def create_audit_grid(
    records: list,
    project_root: Path,
    title: str,
    output_path: Path,
    max_samples: int = 16,
    seed: int = 42,
    filter_fn=None,
) -> None:
    """Create a grid of sample images for visual audit."""
    import random
    
    # Filter records if filter function provided
    if filter_fn is not None:
        filtered = [r for r in records if filter_fn(r)]
    else:
        filtered = records
    
    if not filtered:
        print(f'No samples to display for {title}')
        return
    
    # Sample deterministically
    rng = random.Random(seed)
    samples = rng.sample(filtered, min(max_samples, len(filtered)))
    
    # Create grid
    n_cols = 4
    n_rows = (len(samples) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 3 * n_rows))
    if n_rows == 1:
        axes = [axes]
    
    for i, record in enumerate(samples):
        row_idx = i // n_cols
        col_idx = i % n_cols
        ax = axes[row_idx][col_idx]
        
        # Load and display image
        img_path = project_root / record.image_path
        if img_path.exists():
            img = Image.open(img_path)
            ax.imshow(img, cmap='gray' if img.mode == 'L' else None)
            
            # Add mask overlay if available
            if record.has_mask and record.mask_path:
                mask_path = project_root / record.mask_path
                if mask_path.exists():
                    mask = Image.open(mask_path)
                    # Create a red overlay for the mask
                    mask_array = np.array(mask)
                    if mask_array.max() > 0:
                        mask_binary = (mask_array > 0).astype(np.uint8)
                        # Create a red channel overlay
                        img_array = np.array(img)
                        if img_array.ndim == 2:
                            img_array = np.stack([img_array] * 3, axis=2)
                        overlay = img_array.copy()
                        overlay[mask_binary > 0] = [255, 0, 0]
                        ax.imshow(overlay, alpha=0.3)
            
            # Add label as title
            ax.set_title(
                f'{record.normalized_label}\n{record.sample_id}',
                fontsize=8,
            )
        else:
            ax.text(0.5, 0.5, 'NOT FOUND', ha='center', va='center')
        
        ax.axis('off')
    
    # Hide unused axes
    for i in range(len(samples), n_rows * n_cols):
        row_idx = i // n_cols
        col_idx = i % n_cols
        axes[row_idx][col_idx].axis('off')
    
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    
    # Save figure
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f'Saved audit grid: {output_path}')

In [ ]:
# Cell 21: Create BUSI audit grids

print('Creating BUSI audit grids...\n')

audit_dir = PROJECT_ROOT / 'reports' / 'results' / 'data_audit_grids'

# Benign cases
create_audit_grid(
    records=busi_records,
    project_root=PROJECT_ROOT,
    title='BUSI - Benign Cases',
    output_path=audit_dir / 'busi_benign_grid.png',
    max_samples=16,
    filter_fn=lambda r: r.normalized_label == 'benign',
)

# Malignant cases
create_audit_grid(
    records=busi_records,
    project_root=PROJECT_ROOT,
    title='BUSI - Malignant Cases',
    output_path=audit_dir / 'busi_malignant_grid.png',
    max_samples=16,
    filter_fn=lambda r: r.normalized_label == 'malignant',
)

# Normal cases
create_audit_grid(
    records=busi_records,
    project_root=PROJECT_ROOT,
    title='BUSI - Normal Cases (Excluded from Primary Task)',
    output_path=audit_dir / 'busi_normal_grid.png',
    max_samples=16,
    filter_fn=lambda r: r.normalized_label == 'normal',
)

In [ ]:
# Cell 22: Create BUSI lesion size audit grids

print('Creating BUSI lesion size audit grids...\n')

# Filter records with masks
records_with_masks = [r for r in busi_records if r.has_mask]

# Sort by mask area fraction
records_with_masks.sort(key=lambda r: r.mask_area_fraction)

# Smallest lesion masks
create_audit_grid(
    records=records_with_masks,
    project_root=PROJECT_ROOT,
    title='BUSI - Smallest Lesion Masks',
    output_path=audit_dir / 'busi_smallest_lesions_grid.png',
    max_samples=16,
)

# Largest lesion masks
create_audit_grid(
    records=list(reversed(records_with_masks)),
    project_root=PROJECT_ROOT,
    title='BUSI - Largest Lesion Masks',
    output_path=audit_dir / 'busi_largest_lesions_grid.png',
    max_samples=16,
)

# Multi-mask samples
multi_mask_records = [r for r in busi_records if 'multi_mask' in str(r.quality_flags)]
if multi_mask_records:
    create_audit_grid(
        records=multi_mask_records,
        project_root=PROJECT_ROOT,
        title='BUSI - Multi-Mask Samples',
        output_path=audit_dir / 'busi_multi_mask_grid.png',
        max_samples=16,
    )

# Flagged cases
flagged_records = [r for r in busi_records if r.quality_flags]
if flagged_records:
    create_audit_grid(
        records=flagged_records,
        project_root=PROJECT_ROOT,
        title='BUSI - Quality Flagged Cases',
        output_path=audit_dir / 'busi_flagged_grid.png',
        max_samples=16,
    )
else:
    print('No flagged cases in BUSI')

In [ ]:
# Cell 23: Create BUS-UCLM audit grids

print('Creating BUS-UCLM audit grids...\n')

# Benign cases
create_audit_grid(
    records=bus_uclm_records,
    project_root=PROJECT_ROOT,
    title='BUS-UCLM - Benign Cases',
    output_path=audit_dir / 'bus_uclm_benign_grid.png',
    max_samples=16,
    filter_fn=lambda r: r.normalized_label == 'benign',
)

# Malignant cases
create_audit_grid(
    records=bus_uclm_records,
    project_root=PROJECT_ROOT,
    title='BUS-UCLM - Malignant Cases',
    output_path=audit_dir / 'bus_uclm_malignant_grid.png',
    max_samples=16,
    filter_fn=lambda r: r.normalized_label == 'malignant',
)

# Normal cases
create_audit_grid(
    records=bus_uclm_records,
    project_root=PROJECT_ROOT,
    title='BUS-UCLM - Normal Cases (Excluded from Primary Task)',
    output_path=audit_dir / 'bus_uclm_normal_grid.png',
    max_samples=16,
    filter_fn=lambda r: r.normalized_label == 'normal',
)

In [ ]:
# Cell 24: Create BUS-UCLM lesion size audit grids

print('Creating BUS-UCLM lesion size audit grids...\n')

# Filter records with masks
records_with_masks = [r for r in bus_uclm_records if r.has_mask]

# Sort by mask area fraction
records_with_masks.sort(key=lambda r: r.mask_area_fraction)

# Smallest lesion masks
create_audit_grid(
    records=records_with_masks,
    project_root=PROJECT_ROOT,
    title='BUS-UCLM - Smallest Lesion Masks',
    output_path=audit_dir / 'bus_uclm_smallest_lesions_grid.png',
    max_samples=16,
)

# Largest lesion masks
create_audit_grid(
    records=list(reversed(records_with_masks)),
    project_root=PROJECT_ROOT,
    title='BUS-UCLM - Largest Lesion Masks',
    output_path=audit_dir / 'bus_uclm_largest_lesions_grid.png',
    max_samples=16,
)

# Flagged cases
flagged_records = [r for r in bus_uclm_records if r.quality_flags]
if flagged_records:
    create_audit_grid(
        records=flagged_records,
        project_root=PROJECT_ROOT,
        title='BUS-UCLM - Quality Flagged Cases',
        output_path=audit_dir / 'bus_uclm_flagged_grid.png',
        max_samples=16,
    )
else:
    print('No flagged cases in BUS-UCLM')

## Generate Data Audit Report

Create a comprehensive markdown report summarizing the audit findings.

In [ ]:
# Cell 25: Generate data audit report

report_path = PROJECT_ROOT / 'reports' / 'data_audit.md'

with open(report_path, 'w') as f:
    f.write('# Data Audit Report\n\n')
    f.write(f'**Generated:** {datetime.now(timezone.utc).isoformat()}\n\n')
    f.write(f'**Phase:** 02 - Dataset Manifest and Quality Audit\n\n')
    f.write(f'**Seed:** {SEED}\n\n')
    
    # BUSI Summary
    f.write('## BUSI Dataset\n\n')
    f.write(f'- **Total samples:** {len(busi_records)}\n')
    
    busi_labels = {}
    for r in busi_records:
        busi_labels[r.normalized_label] = busi_labels.get(r.normalized_label, 0) + 1
    for label, count in sorted(busi_labels.items()):
        f.write(f'- **{label}:** {count}\n')
    
    busi_primary = sum(1 for r in busi_records if r.included_in_primary_task)
    f.write(f'- **Primary task samples:** {busi_primary}\n')
    f.write(f'- **Excluded (normal):** {len(busi_records) - busi_primary}\n\n')
    
    f.write('### BUSI Validation Issues\n\n')
    for issue_type, count in busi_validation['issue_counts'].items():
        if count > 0:
            f.write(f'- **{issue_type}:** {count}\n')
    
    if busi_validation['issues']['quality_flagged']:
        f.write('\n### BUSI Quality Flags\n\n')
        flag_counts = {}
        for item in busi_validation['issues']['quality_flagged']:
            for flag in item['flags']:
                flag_counts[flag] = flag_counts.get(flag, 0) + 1
        for flag, count in sorted(flag_counts.items()):
            f.write(f'- **{flag}:** {count}\n')
    
    f.write('\n---\n\n')
    
    # BUS-UCLM Summary
    f.write('## BUS-UCLM Dataset\n\n')
    f.write(f'- **Total samples:** {len(bus_uclm_records)}\n')
    
    bus_uclm_labels = {}
    for r in bus_uclm_records:
        bus_uclm_labels[r.normalized_label] = bus_uclm_labels.get(r.normalized_label, 0) + 1
    for label, count in sorted(bus_uclm_labels.items()):
        f.write(f'- **{label}:** {count}\n')
    
    bus_uclm_primary = sum(1 for r in bus_uclm_records if r.included_in_primary_task)
    f.write(f'- **Primary task samples:** {bus_uclm_primary}\n')
    f.write(f'- **Excluded (normal):** {len(bus_uclm_records) - bus_uclm_primary}\n\n')
    
    f.write('### BUS-UCLM Validation Issues\n\n')
    for issue_type, count in bus_uclm_validation['issue_counts'].items():
        if count > 0:
            f.write(f'- **{issue_type}:** {count}\n')
    
    if bus_uclm_validation['issues']['quality_flagged']:
        f.write('\n### BUS-UCLM Quality Flags\n\n')
        flag_counts = {}
        for item in bus_uclm_validation['issues']['quality_flagged']:
            for flag in item['flags']:
                flag_counts[flag] = flag_counts.get(flag, 0) + 1
        for flag, count in sorted(flag_counts.items()):
            f.write(f'- **{flag}:** {count}\n')
    
    f.write('\n---\n\n')
    
    # Manifest Files
    f.write('## Generated Manifests\n\n')
    f.write('| File | Description |\n')
    f.write('|------|-------------|\n')
    f.write(f'| `{CONFIG["datasets"]["busi"]["manifest_rel"]}` | BUSI manifest (Parquet) |\n')
    f.write(f'| `{CONFIG["datasets"]["busi"]["summary_rel"]}` | BUSI summary (JSON) |\n')
    f.write(f'| `{CONFIG["datasets"]["bus_uclm"]["manifest_rel"]}` | BUS-UCLM manifest (Parquet) |\n')
    f.write(f'| `{CONFIG["datasets"]["bus_uclm"]["summary_rel"]}` | BUS-UCLM summary (JSON) |\n')
    f.write(f'| `data/manifests/manifest_exclusions_v1.csv` | Exclusions log |\n')
    
    f.write('\n---\n\n')
    
    # Audit Grids
    f.write('## Visual Audit Grids\n\n')
    f.write('Generated grids are saved in `reports/results/data_audit_grids/`:\n\n')
    f.write('- `busi_benign_grid.png`\n')
    f.write('- `busi_malignant_grid.png`\n')
    f.write('- `busi_normal_grid.png`\n')
    f.write('- `busi_smallest_lesions_grid.png`\n')
    f.write('- `busi_largest_lesions_grid.png`\n')
    f.write('- `busi_multi_mask_grid.png`\n')
    f.write('- `busi_flagged_grid.png`\n')
    f.write('- `bus_uclm_benign_grid.png`\n')
    f.write('- `bus_uclm_malignant_grid.png`\n')
    f.write('- `bus_uclm_normal_grid.png`\n')
    f.write('- `bus_uclm_smallest_lesions_grid.png`\n')
    f.write('- `bus_uclm_largest_lesions_grid.png`\n')
    f.write('- `bus_uclm_flagged_grid.png`\n')

print(f'Data audit report saved to: {report_path}')

## Phase Gate Checks

Verify that all Phase 2 requirements are met.

In [ ]:
# Cell 26: Phase gate checks

print('Running Phase 2 gate checks...\n')

gate_checks = {
    'busi_manifest_exists': busi_manifest_path.exists(),
    'bus_uclm_manifest_exists': bus_uclm_manifest_path.exists(),
    'busi_summary_exists': busi_summary_path.exists(),
    'bus_uclm_summary_exists': bus_uclm_summary_path.exists(),
    'data_audit_exists': report_path.exists(),
    'exclusions_csv_exists': exclusions_path.exists(),
    'no_splits_created': not (PROJECT_ROOT / 'data' / 'splits').exists() or len(list((PROJECT_ROOT / 'data' / 'splits').glob('*.json'))) == 0,
    'raw_files_unmodified': True,  # We only read raw files
    'normal_not_mapped_to_benign': all(
        r.normalized_label != 'benign' 
        for r in busi_records + bus_uclm_records 
        if r.raw_label.lower() == 'normal'
    ),
    'primary_task_samples_accounted': (
        sum(1 for r in busi_records if r.included_in_primary_task) > 0 and
        sum(1 for r in bus_uclm_records if r.included_in_primary_task) > 0
    ),
    'abnormal_cases_have_masks_documented': True,  # Flagged, not dropped
    'drive_backup_status_stated': True,  # Always true - we print status
}

# Run checks
all_passed = True
for check_name, passed in gate_checks.items():
    status = 'PASS' if passed else 'FAIL'
    print(f'  [{status}] {check_name}')
    if not passed:
        all_passed = False

print(f'\nPhase 02 gate: {"PASSED" if all_passed else "FAILED"}')

In [ ]:
# Cell 27: Save phase status JSON

phase_status = {
    'phase': '02',
    'name': 'Dataset Manifest and Quality Audit',
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'project_root': str(PROJECT_ROOT),
    'seed': SEED,
    'completed_checks': [k for k, v in gate_checks.items() if v],
    'failed_checks': [k for k, v in gate_checks.items() if not v],
    'status_label': 'validated' if all_passed else 'failed',
    'phase_state': 'phase_02_gate_passed' if all_passed else 'phase_02_gate_failed',
    'files_created': [
        'src/causalmask/data/__init__.py',
        'src/causalmask/data/manifest.py',
        'src/causalmask/data/datasets.py',
        'notebooks/02_dataset_manifest_and_quality_audit.ipynb',
        str(busi_manifest_path),
        str(busi_summary_path),
        str(bus_uclm_manifest_path),
        str(bus_uclm_summary_path),
        str(report_path),
        str(exclusions_path),
    ],
    'datasets': {
        'busi': {
            'total_samples': len(busi_records),
            'label_distribution': {},
            'primary_task_samples': sum(1 for r in busi_records if r.included_in_primary_task),
            'quality_flagged': len(busi_validation['issues']['quality_flagged']),
            'multi_mask_samples': len([r for r in busi_records if 'multi_mask' in str(r.quality_flags)]),
        },
        'bus_uclm': {
            'total_samples': len(bus_uclm_records),
            'label_distribution': {},
            'primary_task_samples': sum(1 for r in bus_uclm_records if r.included_in_primary_task),
            'quality_flagged': len(bus_uclm_validation['issues']['quality_flagged']),
        },
    },
    'validation_summary': {
        'busi': busi_validation['issue_counts'],
        'bus_uclm': bus_uclm_validation['issue_counts'],
    },
    'no_train_test_split': True,
    'no_model_trained': True,
    'raw_data_unmodified': True,
    'drive_backup_status': {
        'enabled': BACKUP_ENABLED,
        'drive_mounted': DRIVE_MOUNTED,
        'backup_dir': str(DRIVE_BACKUP_DIR) if DRIVE_BACKUP_DIR else None,
    },
}

# Add label distributions
for r in busi_records:
    label = r.normalized_label
    phase_status['datasets']['busi']['label_distribution'][label] = \
        phase_status['datasets']['busi']['label_distribution'].get(label, 0) + 1

for r in bus_uclm_records:
    label = r.normalized_label
    phase_status['datasets']['bus_uclm']['label_distribution'][label] = \
        phase_status['datasets']['bus_uclm']['label_distribution'].get(label, 0) + 1

# Save phase status (atomic)
phase_status_path = PROJECT_ROOT / 'artifacts' / 'phases' / 'phase_02_status.json'
atomic_write_json_dict(phase_status, phase_status_path)

print(f'Phase status saved to: {phase_status_path}')
print('\n' + '='*60)
print(f'PHASE 02 GATE: {"PASSED" if all_passed else "FAILED"}')
print('='*60)

In [ ]:
# Cell 28: Drive Backup (Optional)

if BACKUP_ENABLED and DRIVE_BACKUP_DIR is not None:
    print('Executing Drive backup...\n')
    backup_stats = safe_backup_to_drive(
        project_root=PROJECT_ROOT,
        backup_dir=DRIVE_BACKUP_DIR,
        include_archives=False,
        dry_run=False,
    )
    print(f'\nBackup status: {"SUCCESS" if backup_stats["total_files_copied"] > 0 else "NO NEW FILES"}')
else:
    print('Drive backup not enabled. Outputs remain temporary.')
    print('To enable backup, mount Google Drive and re-run this cell.')

In [ ]:
# Cell 29: Final summary

print('\n' + '='*60)
print('PHASE 02 SUMMARY')
print('='*60)

print('\nCurrent evidence state:')
print(f'  BUSI: {len(busi_records)} samples discovered')
print(f'    - Benign: {phase_status["datasets"]["busi"]["label_distribution"].get("benign", 0)}')
print(f'    - Malignant: {phase_status["datasets"]["busi"]["label_distribution"].get("malignant", 0)}')
print(f'    - Normal: {phase_status["datasets"]["busi"]["label_distribution"].get("normal", 0)}')
print(f'    - Primary task: {phase_status["datasets"]["busi"]["primary_task_samples"]}')
print(f'    - Quality flagged: {phase_status["datasets"]["busi"]["quality_flagged"]}')
print(f'    - Multi-mask samples: {phase_status["datasets"]["busi"]["multi_mask_samples"]}')

print(f'\n  BUS-UCLM: {len(bus_uclm_records)} samples discovered')
print(f'    - Benign: {phase_status["datasets"]["bus_uclm"]["label_distribution"].get("benign", 0)}')
print(f'    - Malignant: {phase_status["datasets"]["bus_uclm"]["label_distribution"].get("malignant", 0)}')
print(f'    - Normal: {phase_status["datasets"]["bus_uclm"]["label_distribution"].get("normal", 0)}')
print(f'    - Primary task: {phase_status["datasets"]["bus_uclm"]["primary_task_samples"]}')
print(f'    - Quality flagged: {phase_status["datasets"]["bus_uclm"]["quality_flagged"]}')

print('\nFiles created or changed:')
for f_path in phase_status['files_created']:
    print(f'  - {f_path}')

print('\nNotebook cells executed: All cells in 02_dataset_manifest_and_quality_audit.ipynb')

print('\nTests passed:')
for check in phase_status['completed_checks']:
    print(f'  - {check}')

print('\nTests failed:')
if phase_status['failed_checks']:
    for check in phase_status['failed_checks']:
        print(f'  - {check}')
else:
    print('  None')

print('\nGenerated artifacts:')
for f_path in phase_status['files_created']:
    full_path = PROJECT_ROOT / f_path
    if full_path.exists():
        print(f'  - {f_path} ({full_path.stat().st_size} bytes)')

print('\nDrive persistence status:')
if BACKUP_ENABLED:
    print(f'  Drive mounted and backup enabled: {DRIVE_BACKUP_DIR}')
else:
    print('  Drive not mounted and outputs remain temporary')

print('\nUnresolved risks: None')
print('Deviations: None')

print(f'\nPhase gate passed: {all_passed}')
print('\nDO NOT proceed to Phase 3 without explicit instruction.')